In [ ]:
from huggingface_hub import HfApi, login

login()

api =   HfApi()

repo_id = "ShanmukhVashtav/Fineweb-edu-5B-gpt-2-tokenized "


In [ ]:
from datasets import load_dataset
import tiktoken
import os

TARGET_TOKENS = 5_000_000_000
SHARD_SIZE = 1_000_000_000

output_dir = "fineweb_ed_5b"

os.makedirs(output_dir, exist_ok=True)

ds = load_dataset("HuggingFaceFW/fineweb-edu", streaming=True, name="sample-10BT", split="train")

enc = tiktoken.get_encoding("gpt2")

In [ ]:
from tqdm.auto import tqdm
import numpy as np
buffer = []
total_tokens = 0
shard_id = 0


pbar = tqdm(total=TARGET_TOKENS, units="tokens")

for sample in ds:
    text = sample["text"]

    buffer.extend(enc.encode(text))


    while len(buffer) >= SHARD_SIZE:
        shard_tokens = np.asarray(
            buffer[:SHARD_SIZE],
            dtype=np.uint16
        )


        path = os.path.join(output_dir, f"shard_{shard_id:02d}.npy")


        np.save(path, shard_tokens)

        api.upload_file(
            path_or_fileobj=path,
            path_in_repo=f"train/shard_{shard_id:02d}.npy",
            repo_id=repo_id,
            repo_type="dataset"
        )


        total_tokens += SHARD_SIZE
        shard_id += 1

        pbar.update(SHARD_SIZE)

        print(
            f"Saved shard {shard_id}: "
            f"{total_tokens:,} / {TARGET_TOKENS:,} tokens"
        )


        # print(f"Shard {shard_id:02d} uploaded")

        buffer = buffer[SHARD_SIZE:]

        if total_tokens >= TARGET_TOKENS:
            break
    if total_tokens >= TARGET_TOKENS:
        break

pbar.close()

print("Completed!")
